# FL-BENCH-004.7 — Leaderboard

*Epic EPIC-001 · story fl_bench_004 · all results below are loaded from files produced by the task's experiment function and labelled MEASURED/SIMULATED; clients are virtual; data is synthetic.*

## 1. Research objective
Multi-criteria leaderboard (utility, convergence, communication, robustness, cost).

## 2. Scientific background
Mean-rank aggregation of per-criterion ranks [demsar2006].

References use keys of `docs/references/references_master.md` (status per entry).

## 3. Research questions
- **RQ1.** Does the implementation satisfy its validation identities?
- **RQ2.** How does it behave under IID vs non-IID data?

## 4. Hypotheses
- **H1.** Validation identities hold to numerical precision.

## 5. Mathematical formulation
mean rank = average over criteria of rank (1 = best).

## 6. Experimental environment

In [1]:
import os, sys, json, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
def _find_root():
    p = Path(os.environ.get("FL_ROOT") or os.environ.get("RS_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set RS_ROOT")
ROOT = _find_root()
sys.path.insert(0, str(ROOT / "src"))
STORY = "fl_bench_004"
_cands = [ROOT / "experiments" / "results" / STORY]
RES = next(c for c in _cands if c.exists())
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
SEED = 0; np.random.seed(SEED)
def show_csv(name, n=12):
    p = RES / name
    if p.exists(): display(pd.read_csv(p).round(4).head(n))
    else: print("MISSING", name)
def show_fig(name):
    for base in (RES / "figures", RES):
        if (base / name).exists(): display(Image(filename=str(base / name))); return
    print("MISSING figure", name)
print("root:", ROOT, "| results:", RES)

root: /home/claude/fl_sprint | results: /home/claude/fl_sprint/experiments/results/fl_bench_004


In [2]:
import importlib.metadata as md
vers = {}
for p in ["numpy","scipy","scikit-learn","torch","tenseal","phe","pandas","matplotlib","cryptography","psutil","mlflow","opacus"]:
    try: vers[p] = md.version(p)
    except Exception: vers[p] = "not installed"
print(json.dumps(dict(python=platform.python_version(), platform=platform.platform(), versions=vers, seed=SEED, gpu="none (CPU only; GPU metrics NOT_MEASURED)"), indent=1))

{
 "python": "3.13.18",
 "platform": "Linux-6.18.44-fc-v338-x86_64-with-glibc2.39",
 "versions": {
  "numpy": "2.5.3",
  "scipy": "1.18.1",
  "scikit-learn": "1.9.1",
  "torch": "2.15.0",
  "tenseal": "0.3.18",
  "phe": "1.5.0",
  "pandas": "3.0.5",
  "matplotlib": "3.11.2",
  "cryptography": "50.0.1",
  "psutil": "7.2.2",
  "mlflow": "3.18.0",
  "opacus": "1.6.0"
 },
 "seed": 0,
 "gpu": "none (CPU only; GPU metrics NOT_MEASURED)"
}


## 7. Dataset description
Synthetic binary tabular dataset **SYN-IDS-v1** (scikit-learn `make_classification`, 40 features, 70:30 split). Chosen because no real IDS/clinical dataset was available in the sandbox. It has no clinical meaning; conclusions about absolute accuracy do not transfer.

In [3]:
from federated_learning.synthetic_data.generators.syn_ids_v1 import make_dataset, DATASET_ID
from federated_learning.data_partitioning.iid import partition_iid
from federated_learning.data_partitioning.dirichlet import partition_dirichlet
from federated_learning.data_partitioning.metrics import label_skew_stat
data = make_dataset(n_samples=6000, seed=SEED)
print(DATASET_ID, "train", data.X_train.shape, "test", data.X_test.shape, "positive rate", float(data.y_train.mean()))
print("NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.")

SYN-IDS-v1 train (4200, 40) test (1800, 40) positive rate 0.40095238095238095
NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.


## 8. Configuration and reproducibility settings
Seeds are fixed; each stored result carries `provenance` (experiment id, SHA-256 of the configuration, environment versions; `git_commit` is `null` because the workspace is not a git repository).

In [4]:
from federated_learning.training.config import FLConfig
cfg = FLConfig(rounds=5, local_epochs=1, seed=SEED)
print(cfg)

FLConfig(algo='fedavg', rounds=5, local_epochs=1, lr=0.05, batch_size=32, participation=1.0, mu=0.0, hidden=32, act='relu', seed=0, hetero_epochs=False, eval_every=1, server_lr=1.0)


## 9. Data preparation
Partitioning into virtual clients (IID or Dirichlet label skew) is performed by `federated_learning.data_partitioning`.

In [5]:
parts_iid = partition_iid(data.y_train, 10, SEED)
parts_dir = partition_dirichlet(data.y_train, 10, 0.3, SEED)
print('label-skew statistic IID:', round(label_skew_stat(data.y_train, parts_iid),3), ' Dirichlet(0.3):', round(label_skew_stat(data.y_train, parts_dir),3))

label-skew statistic IID: 0.019  Dirichlet(0.3): 0.363


## 10. Implementation
Reusable code lives in `src/federated_learning/` (`fl.py`, `stats.py`). The cell below exercises it live (small, fast, deterministic) and asserts correctness.

In [6]:
from federated_learning.training.config import FLConfig
from federated_learning.servers.orchestrator import run_fl
parts = partition_dirichlet(data.y_train, 8, 0.3, SEED)
r = run_fl(FLConfig(algo="fedavg", rounds=6, local_epochs=2, mu=0.05, seed=SEED), data, parts)
print("fedavg final", {k: round(v, 4) for k, v in r.final.items() if isinstance(v, float)}, "comm MB", round(r.totals["comm_mb"], 3))
assert np.isfinite(r.w).all()

fedavg final {'round_time_s': 0.0759, 'agg_time_s': 0.0003, 'client_time_s': 0.0755, 'client_time_max_s': 0.0335, 'acc': 0.7911, 'loss': 0.4412, 'precision': 0.8165, 'recall': 0.6172, 'f1': 0.703, 'specificity': 0.9073, 'auc': 0.8767, 'diverged': 0.0} comm MB 0.505


## 11. Experiment execution
The full experiment was executed by `python -m federated_learning.pipelines.experiments.run_all --task FL-BENCH-004.7` (see `logs/`). Its raw per-run results are loaded here; they are not re-computed inside the notebook.

In [7]:
show_csv('task_4_7_leaderboard_iid.csv')
show_csv('task_4_7_leaderboard_dir0.3.csv')

,algorithm,key,utility_f1,convergence_rounds,communication_mb_to_target,robustness_acc_under_signflip,training_cost_s,rank_utility_f1,rank_convergence_rounds,rank_communication_mb_to_target,rank_robustness_acc_under_signflip,rank_training_cost_s,mean_rank,note
0,FedAvg,fedavg,0.8840,11.8,1.2406,0.6609,8.2729,2.0,1.0,1.0,2.0,1.0,1.4,NaN
1,SCAFFOLD,scaffold,0.8849,11.8,2.4811,0.8147,8.3072,1.0,2.0,4.0,1.0,2.0,2.0,NaN
2,FedNova,fednova,0.8840,11.8,1.2406,0.6229,8.3269,3.0,3.0,2.0,4.0,3.0,3.0,NaN
3,FedProx,fedprox,0.8824,12.6,1.3247,0.6585,8.5312,4.0,4.0,3.0,3.0,4.0,3.6,NaN
4,Proposed framework (FedAvg + CKKS HE aggregation),proposed,0.8841,11.8,37.9104,NaN,11.2263,NaN,NaN,NaN,NaN,NaN,NaN,not ranked: robustness dimension NOT_MEASURED ...


,algorithm,key,utility_f1,convergence_rounds,communication_mb_to_target,robustness_acc_under_signflip,training_cost_s,rank_utility_f1,rank_convergence_rounds,rank_communication_mb_to_target,rank_robustness_acc_under_signflip,rank_training_cost_s,mean_rank,note
0,FedAvg,fedavg,0.8431,16.8,1.7662,0.6609,7.9543,3.0,3.0,2.0,2.0,1.0,2.2,NaN
1,SCAFFOLD,scaffold,0.8729,11.4,2.3970,0.8147,8.1876,1.0,1.0,4.0,1.0,4.0,2.2,NaN
2,FedNova,fednova,0.8637,12.8,1.3457,0.6229,8.0003,2.0,2.0,1.0,4.0,2.0,2.2,NaN
3,FedProx,fedprox,0.8391,18.0,1.8924,0.6585,8.0590,4.0,4.0,3.0,3.0,3.0,3.4,NaN
4,Proposed framework (FedAvg + CKKS HE aggregation),proposed,0.8431,16.8,53.9809,NaN,11.2549,NaN,NaN,NaN,NaN,NaN,NaN,not ranked: robustness dimension NOT_MEASURED ...


## 12. Metrics collection
Metric definitions: accuracy, F1, rounds-to-target (acc≥0.85), logical communication MB, local drift ||w_k − w_t||.

In [8]:
show_csv('task_4_7_leaderboard_dir0.3.csv')

,algorithm,key,utility_f1,convergence_rounds,communication_mb_to_target,robustness_acc_under_signflip,training_cost_s,rank_utility_f1,rank_convergence_rounds,rank_communication_mb_to_target,rank_robustness_acc_under_signflip,rank_training_cost_s,mean_rank,note
0,FedAvg,fedavg,0.8431,16.8,1.7662,0.6609,7.9543,3.0,3.0,2.0,2.0,1.0,2.2,NaN
1,SCAFFOLD,scaffold,0.8729,11.4,2.3970,0.8147,8.1876,1.0,1.0,4.0,1.0,4.0,2.2,NaN
2,FedNova,fednova,0.8637,12.8,1.3457,0.6229,8.0003,2.0,2.0,1.0,4.0,2.0,2.2,NaN
3,FedProx,fedprox,0.8391,18.0,1.8924,0.6585,8.0590,4.0,4.0,3.0,3.0,3.0,3.4,NaN
4,Proposed framework (FedAvg + CKKS HE aggregation),proposed,0.8431,16.8,53.9809,NaN,11.2549,NaN,NaN,NaN,NaN,NaN,NaN,not ranked: robustness dimension NOT_MEASURED ...


## 13. Statistical analysis
Seeds are the unit of replication; paired tests across seeds (Wilcoxon signed-rank, paired t), Holm correction, Cohen's d_z; see task 4.6.

In [9]:
show_csv('task_4_7_leaderboard_dir0.3.csv', 30)

,algorithm,key,utility_f1,convergence_rounds,communication_mb_to_target,robustness_acc_under_signflip,training_cost_s,rank_utility_f1,rank_convergence_rounds,rank_communication_mb_to_target,rank_robustness_acc_under_signflip,rank_training_cost_s,mean_rank,note
0,FedAvg,fedavg,0.8431,16.8,1.7662,0.6609,7.9543,3.0,3.0,2.0,2.0,1.0,2.2,NaN
1,SCAFFOLD,scaffold,0.8729,11.4,2.3970,0.8147,8.1876,1.0,1.0,4.0,1.0,4.0,2.2,NaN
2,FedNova,fednova,0.8637,12.8,1.3457,0.6229,8.0003,2.0,2.0,1.0,4.0,2.0,2.2,NaN
3,FedProx,fedprox,0.8391,18.0,1.8924,0.6585,8.0590,4.0,4.0,3.0,3.0,3.0,3.4,NaN
4,Proposed framework (FedAvg + CKKS HE aggregation),proposed,0.8431,16.8,53.9809,NaN,11.2549,NaN,NaN,NaN,NaN,NaN,NaN,not ranked: robustness dimension NOT_MEASURED ...


## 14. Visualizations

In [10]:
print('no figure for this task (tabular result)')

no figure for this task (tabular result)


## 15. Interpretation of results
The numbers are computed in the next cell from the stored data; the text interprets only what that cell prints.

Rankings differ between IID and non-IID; the CKKS row is not ranked (robustness NOT_MEASURED).

In [11]:
show_csv('task_4_7_leaderboard_iid.csv'); show_csv('task_4_7_leaderboard_dir0.3.csv')

,algorithm,key,utility_f1,convergence_rounds,communication_mb_to_target,robustness_acc_under_signflip,training_cost_s,rank_utility_f1,rank_convergence_rounds,rank_communication_mb_to_target,rank_robustness_acc_under_signflip,rank_training_cost_s,mean_rank,note
0,FedAvg,fedavg,0.8840,11.8,1.2406,0.6609,8.2729,2.0,1.0,1.0,2.0,1.0,1.4,NaN
1,SCAFFOLD,scaffold,0.8849,11.8,2.4811,0.8147,8.3072,1.0,2.0,4.0,1.0,2.0,2.0,NaN
2,FedNova,fednova,0.8840,11.8,1.2406,0.6229,8.3269,3.0,3.0,2.0,4.0,3.0,3.0,NaN
3,FedProx,fedprox,0.8824,12.6,1.3247,0.6585,8.5312,4.0,4.0,3.0,3.0,4.0,3.6,NaN
4,Proposed framework (FedAvg + CKKS HE aggregation),proposed,0.8841,11.8,37.9104,NaN,11.2263,NaN,NaN,NaN,NaN,NaN,NaN,not ranked: robustness dimension NOT_MEASURED ...


,algorithm,key,utility_f1,convergence_rounds,communication_mb_to_target,robustness_acc_under_signflip,training_cost_s,rank_utility_f1,rank_convergence_rounds,rank_communication_mb_to_target,rank_robustness_acc_under_signflip,rank_training_cost_s,mean_rank,note
0,FedAvg,fedavg,0.8431,16.8,1.7662,0.6609,7.9543,3.0,3.0,2.0,2.0,1.0,2.2,NaN
1,SCAFFOLD,scaffold,0.8729,11.4,2.3970,0.8147,8.1876,1.0,1.0,4.0,1.0,4.0,2.2,NaN
2,FedNova,fednova,0.8637,12.8,1.3457,0.6229,8.0003,2.0,2.0,1.0,4.0,2.0,2.2,NaN
3,FedProx,fedprox,0.8391,18.0,1.8924,0.6585,8.0590,4.0,4.0,3.0,3.0,3.0,3.4,NaN
4,Proposed framework (FedAvg + CKKS HE aggregation),proposed,0.8431,16.8,53.9809,NaN,11.2549,NaN,NaN,NaN,NaN,NaN,NaN,not ranked: robustness dimension NOT_MEASURED ...


## 16. Limitations
- Synthetic tabular data; small MLP.
- Synthetic data, virtual clients on one machine, 2 CPU cores, no GPU; no real network. Comm. bytes are logical payload sizes.
- No regulatory/compliance conclusion can be drawn from this notebook.

## 17. Conclusions
Implementation validated by identity tests (tests/ and task JSON).

**Jira acceptance:** see `docs/fl_bench_004/results_and_discussion.md` for the criterion-by-criterion verdict (MET / PARTIALLY MET / NOT MET).

## 18. References
`mcmahan2017`, `li2020fedprox`, `karimireddy2020`, `wang2020fednova`, `li2021niid` — details in `docs/fl_bench_004/references.md`.